In [5]:
#Loading necessary libraries
import numpy as np
import pandas as pd
from pathlib import Path
import tensorflow as tf
import matplotlib.pyplot as plt
import json


In [6]:
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32

train_df = pd.read_csv("../data/processed/train_split.csv")
val_df   = pd.read_csv("../data/processed/val_split.csv")
test_df  = pd.read_csv("../data/processed/test_split.csv")

print(len(train_df), len(val_df), len(test_df))

99582 21339 21340


In [7]:
with open("../artifacts/class_mapping.json", "r") as f:
    mapping = json.load(f)

classes = mapping["classes"]
num_classes = len(classes)
print("Classes:", num_classes)


Classes: 36


In [8]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=1)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def make_dataset(df, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label_id"].values))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000, seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df, shuffle=False)
test_ds  = make_dataset(test_df, shuffle=False)


In [9]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomContrast(0.1),
], name="augmentation")


In [10]:
model2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64, 64, 1)),
    data_augmentation,

    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.15),

    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model2.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 36)             │         9,252 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,200,228 (8.39 MB)

 Trainable params: 2,199,780 (8.39 MB)

 Non-trainable params: 448 (1.75 KB)

In [11]:
model2.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5)
]


In [12]:
history2 = model2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks
)


Epoch 1/8
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 774s 245ms/step - accuracy: 0.7059 - loss: 0.9386 - val_accuracy: 0.7975 - val_loss: 0.7095 - learning_rate: 0.0010
Epoch 2/8
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 1135s 361ms/step - accuracy: 0.8758 - loss: 0.3626 - val_accuracy: 0.9500 - val_loss: 0.1519 - learning_rate: 0.0010
Epoch 3/8
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 896s 287ms/step - accuracy: 0.9149 - loss: 0.2576 - val_accuracy: 0.9715 - val_loss: 0.0789 - learning_rate: 0.0010
Epoch 4/8
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 977s 312ms/step - accuracy: 0.9334 - loss: 0.2025 - val_accuracy: 0.9768 - val_loss: 0.0698 - learning_rate: 0.0010
Epoch 5/8
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 977s 310ms/step - accuracy: 0.9470 - loss: 0.1643 - val_accuracy: 0.9754 - val_loss: 0.0694 - learning_rate: 0.0010
Epoch 6/8
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 930s 298ms/step - accuracy: 0.9552 - loss: 0.1421 - val_accuracy: 0.9870 - val_loss: 0.0386 - learning_rate: 0.0010
Epoch 7/8
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 645s 204ms/step 

In [13]:
test_loss2, test_acc2 = model2.evaluate(test_ds)
print("Model 2 Test Accuracy:", test_acc2)


667/667 ━━━━━━━━━━━━━━━━━━━━ 80s 119ms/step - accuracy: 0.9906 - loss: 0.0280
Model 2 Test Accuracy: 0.9906279444694519


In [14]:
from datetime import datetime
import json
from pathlib import Path

MODELS_DIR = Path("../models"); MODELS_DIR.mkdir(exist_ok=True)
ART_DIR = Path("../artifacts"); ART_DIR.mkdir(exist_ok=True)

run_id2 = datetime.now().strftime("%Y%m%d_%H%M%S")
model2.save(MODELS_DIR / "model2_augmented.keras")

hist_df2 = pd.DataFrame(history2.history)
hist_df2["epoch"] = range(1, len(hist_df2)+1)
hist_df2["model_version"] = "model2_augmented"
hist_df2["run_id"] = run_id2
hist_df2.to_csv(ART_DIR / f"train_history_model2_{run_id2}.csv", index=False)

model_card2 = {
    "model_version": "model2_augmented",
    "image_size": list(IMAGE_SIZE),
    "notes": "CNN with augmentation, batch norm, dropout, early stopping."
}
with open(ART_DIR / f"model_card_model2_{run_id2}.json", "w") as f:
    json.dump(model_card2, f, indent=2)

print("Saved Model 2 + artifacts")


Saved Model 2 + artifacts
